In [5]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

In [4]:
import os

from dotenv import find_dotenv, load_dotenv
from openai import OpenAI

load_dotenv(find_dotenv(), override=True)

client = OpenAI(
    api_key=os.getenv("API_KEY"),
    base_url=os.getenv("BASE_URL"),
)


def call_model(system_prompt: str, user_prompt: str, video_b64: str | None = None):
    print("🤖 正在调用多模态模型生成内容...")
    user_content: list[dict] = [
        {
            "type": "text",
            "text": user_prompt,
        }
    ]
    if video_b64:
        user_content.append(
            {
                "type": "video_url",
                "video_url": {"url": f"data:video/mp4;base64,{video_b64}"},
            }
        )

    try:
        response = client.chat.completions.create(
            model=os.getenv("MODEL"),  # type: ignore
            messages=[
                {
                    "role": "system",
                    "content": system_prompt,
                },
                {
                    "role": "user",
                    "content": user_content,
                },  # type: ignore
            ],
        )
    except Exception as e:
        print(f"⚠️ 模型调用失败: {e}")
        raise e

    return response.choices[0].message.content


In [ ]:
from pathlib import Path

from scenedetect import ContentDetector, detect, split_video_ffmpeg

video_path = Path("../tests/videos/6.mp4")
scene_list = detect(video_path, ContentDetector())
split_video_ffmpeg(video_path, scene_list, output_dir=Path("."))

0

In [1]:
SYSTEM_PROMPT = "你是一个视频剪辑专家，擅长根据用户需求对视频进行剪辑。"
USER_PROMPT = "请分析这个视频"

In [2]:
from src.video import VideoClip

video = VideoClip(video_path)

response = call_model(
    system_prompt="你是一个顶级剪辑师，现在你要分析并总结一个视频中的剪辑技巧",
    user_prompt="请分析这个视频",
    video_b64=video.to_base64(),
)

ModuleNotFoundError: No module named 'src'

In [9]:
print(response)

### 一、这个视频用到的核心剪辑技巧拆解
这是典型的「极简文字流情绪向剪辑」，全程靠字幕动态节奏传递信息，没有实拍素材，所有技巧都服务于「突出重点、强化记忆、卡点拉情绪」：
1.  **黑白闪切转场**
    反复用「纯黑底→0.2秒全白闪屏→切回黑底」的硬过渡，完全卡背景音乐重音，不用多余转场特效就造出极强的冲击感，快速把观众注意力拽回内容上。
2.  **分层高亮字幕设计**
    普通铺垫文字用低饱和浅灰色，核心关键词比如“傻子”“11亿”单独拎出来做红底+柔光外发光，视觉权重直接和普通文字拉开差距，观众第一眼就会抓到重点信息，不会被次要内容分散注意力。
3.  **RGB色差故障特效（Glitch）**
    核心反转点的“世界”两个字，做了三原色通道错位散开的复古信号故障效果，只保留0.5秒的闪屏时长，瞬间把反差感拉满，让结尾核心结论的记忆点直接翻倍。
4.  **逐帧关键帧元素入场**
    “友谊、时光、自由”三个吊牌元素，不是直接跳出来，而是依次从画面上方缓慢掉落、小幅度左右摆动，红色吊绳同步跟随卡片运动，节奏和BGM鼓点完全贴合，静态文字瞬间有了生动的动态感。
5.  **文字呼吸动效**
    所有居中的主标题文字，全程做缓慢的「大小缩放+外发光明暗变化」循环，字幕不是静止卡死在画面里，柔和的动态感让全程黑底的画面完全不会沉闷。
6.  **快扫平移信息卡点**
    中间列举11条珍贵事物的段落，所有文字从右往左快速扫过，卡每一拍鼓点放大对应条目的关键词，极快速度铺完所有信息量，节奏紧凑完全不拖沓。

---

### 二、手把手复刻操作步骤
#### 前期准备
先把文案按情绪分段：铺垫段→设问段→列举段→反转段→结论重复段，数清楚BGM每个鼓点的时间节点，字体选厚重的粗衬线/哥特风字体，避开单薄的常规黑体。
1.  基础转场搭建：在PR/AE里新建纯黑、纯白两个固态层，闪白转场的时长控制在0.2-0.3秒，精准卡在BGM重音处裁切，不要做过长的渐变，避免晃眼。
2.  分层字幕制作：普通铺垫文字给70%透明度浅灰色，核心关键词单独新建文字层，填充正红色，添加30%柔和度的外发光效果，把重点字和普通文字错开排布，自动形成视觉重心。
3.  吊牌动画制作：把三个关键词分别放在白色矩形卡片上，给卡片开启3D属性，打「从上掉落+小幅度左